In [1]:
# ============================================================================
# SECTION 1: Import Required Libraries
# ============================================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.io import wavfile
import librosa
import librosa.display
from scipy.signal import lfilter
from scipy.signal.windows import hamming
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
# ============================================================================
# SECTION 3: LPC Coefficient Calculation Functions
# ============================================================================

def autocorr(x, lag=20):
    """
    Calculate autocorrelation using FFT method
    """
    n = len(x)
    x = x - np.mean(x)
    r = np.correlate(x, x, mode='full')[-n:]
    result = r / (n * np.var(x))
    return result[:lag+1]

def levinson_durbin(r, order):
    """
    Levinson-Durbin recursion for LPC coefficient calculation
    """
    a = np.zeros(order + 1)
    e = np.zeros(order + 1)
    a[0] = 1.0
    e[0] = r[0]
    
    for i in range(1, order + 1):
        lambda_val = sum(a[j] * r[i-j] for j in range(i))
        k = -lambda_val / e[i-1]
        
        a_new = np.zeros(order + 1)
        a_new[0] = 1.0
        for j in range(1, i):
            a_new[j] = a[j] + k * a[i-j]
        a_new[i] = k
        a = a_new
        e[i] = (1 - k**2) * e[i-1]
    
    return a, e[-1]

def lpc_analysis(frame, order=12):
    """
    Perform LPC analysis on a speech frame
    Returns LPC coefficients and prediction error
    """
    # Apply Hamming window
    windowed = frame * hamming(len(frame))
    
    # Calculate autocorrelation
    r = autocorr(windowed, order)
    
    # Apply Levinson-Durbin algorithm
    lpc_coeffs, error = levinson_durbin(r, order)
    
    return lpc_coeffs, error

print("\nLPC Analysis Functions defined successfully!")


LPC Analysis Functions defined successfully!
